## BMLL Notebook Educational Series
### Consolidating Best Bid/Offer Across Multiple Venues

In [1]:
import os
from functools import lru_cache
import sys

def install_snowflake_connector():
    os.system(sys.executable + " -m pip install snowflake-connector-python[pandas] snowflake-sqlalchemy")

@lru_cache(5)
def snowflake_connection(
    database="PROD_DERIVED_DATA_DB",
    schema="PUBLIC",
    warehouse="PROD_DERIVED_DEFAULT_WH",
    role="PROD_SALES_RO_ROLE",
    user=None,
    password=None,
    use_secret_storage=True,
    sso=None,
):
    import snowflake
    import snowflake.connector
    try:
        from snowflake.snowpark import Session as SnowparkSession
    except Exception:
        SnowparkSession = None
    try:
        import bmll2
        stor = bmll2.storage_paths()["user"]
        os.environ["SNOWFLAKE_USER"] = stor["prefix"].split("/")[1]
        os.environ["SNOWFLAKE_USE_SSO"] = "1"
    except Exception:
        pass
    if sso is None:
        sso = os.environ.get("SNOWFLAKE_USE_SSO")

    password = os.environ.get("SNOWFLAKE_PASSWORD", password)
    if not password:
        try:
            import secretstorage
            connection = secretstorage.dbus_init()
            collection = secretstorage.get_default_collection(connection)
            for item in collection.search_items({"name": "default_snowflake", "system": "bmll"}):
                password = item.get_secret()
        except Exception as e:
            print(f"Error connecting to Secret Service: {e}")
    else:
        try:
            import secretstorage
            attributes = {"name": "default_snowflake", "system": "bmll"}
            label = "default_snowflake"
            secret = password
            connection = secretstorage.dbus_init()
            collection = secretstorage.get_default_collection(connection)
            if collection.is_locked():
                unlocked = collection.unlock()
                if unlocked:
                    print("Collection unlocked successfully.")
                else:
                    print("Failed to unlock the collection.")
            else:
                print("Collection is already unlocked.")
            collection.create_item(label, attributes, secret, replace=True)
        except Exception as e:
            print(e)

    if not sso:
        connargs = dict(
            user=os.environ.get("SNOWFLAKE_USER", user),
            password=password,
            account=os.environ.get("SNOWFLAKE_ACCOUNT", "tib96089"),
            region=os.environ.get("SNOWFLAKE_REGION", "us-east-1"),
            role=role,
            database=database,
            schema=schema,
            warehouse=warehouse,
        )
    else:
        connargs = dict(
            user=os.environ.get("SNOWFLAKE_USER", user),
            account=os.environ.get("SNOWFLAKE_ACCOUNT", "tib96089"),
            region=os.environ.get("SNOWFLAKE_REGION", "us-east-1"),
            role=role,
            database=database,
            schema=schema,
            warehouse=warehouse,
            authenticator="externalbrowser",
        )
    if SnowparkSession is not None:
        return SnowparkSession.builder.configs(connargs).create()
    return snowflake.connector.connect(**connargs)

session = snowflake_connection()
if not hasattr(session, "sql"):
    class _SFResult:
        def __init__(self, cursor):
            self._cursor = cursor

        def to_pandas(self):
            try:
                return self._cursor.fetch_pandas_all()
            except Exception:
                import pandas as pd
                rows = self._cursor.fetchall()
                cols = [c[0] for c in self._cursor.description] if self._cursor.description else []
                return pd.DataFrame(rows, columns=cols)

    class _SFSessionShim:
        def __init__(self, conn):
            self._conn = conn

        def sql(self, query):
            cur = self._conn.cursor()
            cur.execute(query)
            return _SFResult(cur)

    session = _SFSessionShim(session)

/home/bertrand/miniconda3/lib/python3.11/site-packages/bmll2/_internals/api/configure.py:116: UserWarning: BMLL configuration file (/tmp/.nb_extension_instance_vars.json) not found. BMLL data and S3 storage will not be reachable.
  warnings.warn(f"{msg} BMLL data and S3 storage will not be reachable.", UserWarning)


Initiating login request with your identity provider. A browser window should have opened for you to complete the login. If you can't see it, check existing browser windows, or your OS settings. Press CTRL+C to abort and try again...
Going to open: https://accounts.google.com/o/saml2/idp?idpid=C0433grw8&SAMLRequest=lZJfb9owFMW%2FSuQ9J3YCa8EiIAbrSqEdKmGb9mYcEywcO%2FN1CO2nn8MfqXtopT0kipxz%2FLv3njsYHUsVHIQFaXSK4oigQGhucqmLFK2zu7CHAnBM50wZLVL0IgCNhgNgparouHY7%2FSz%2B1AJc4C%2FSQNsfKaqtpoaBBKpZKYA6TlfjxwVNIkIZgLDO49DFkoP0rJ1zFcW4aZqo6UTGFjghhGDSx17VSj6hN4jqY0ZljTPcqKvl6Ht6BxFj0m0RXuEJy4vxi9TnEXxE2ZxFQO%2BzbBkuv68yFIyv3U2MhroUdiXsQXKxfl6cCwBfwaZUKmxfTvBdBNo0W8X2gpuyqp2%2FMfJfeCtyrEwh%2FZxm0xRVe5mvv8byyVRHOe8eHh52fdgc5vHrZv6tdFX2q6P2SnR%2B3i8eteYcBT%2BuqSZtqjOAWsx0m6XzRyS5CUkvJLdZ3KWf%2B7RLotub%2Fm8UTH2WUjN3cl4LZpybWjuICmMKJU71GdwGkWCZVyP%2FyDydkG6nU9imh84LQk9MO%2Fyftgf4rfOyZ09%2B9LPp0ijJX4I7Y0vm3k8mjuLTiczD7UlKRcmkGue5FQA%2BIaVMM7GCOb%2FOztYC4eGZ%2Bu9CD%2F8C&RelayState=ver%3A3-hint%3A10767547

In [2]:
ticker = 'VOD'
mic = 'XLON'
start_date = '2025-01-01'
end_date = '2025-01-10'

### Why Consolidation Matters

**Multi-Venue Trading Reality**: Many securities trade on multiple exchanges simultaneously:
- **Primary exchange**: Original listing venue (highest volume, best prices typically)
- **Alternative venues**: MTFs (Multilateral Trading Facilities), dark pools, lit venues
- **SIP/Consolidated feeds**: Regulatory consolidated quotes (where available)

**The Consolidation Problem**: Best prices may differ across venues:
- Venue A: Bid $100.05, Ask $100.10
- Venue B: Bid $100.04, Ask $100.09
- Venue C: Bid $100.06, Ask $100.11
- **Consolidated Best (CBBO)**: Bid $100.06 (best), Ask $100.09 (best)

**Why It Matters for Analysis**:
- Traders execute against the **best available prices** (CBBO), not single-venue prices
- Regulatory requirements often reference consolidated quotes
- Market impact analysis needs to account for fragmented liquidity
- Order routing algorithms consider multi-venue depth

**This Notebook**: Demonstrates how to consolidate per-venue best bid/offer (BBO) quotes
into a single consolidated best bid/offer (CBBO) time series using BMLL data.

BMLL provides Level 1 (BBO) quotes from 100+ exchanges. This notebook shows how to:
1. Retrieve BBO for each venue where a security trades
2. Align quotes across venues to the same timestamp
3. Compute consolidated best bid (max bid across venues)
4. Compute consolidated best ask (min ask across venues)
5. Track which venue(s) have the best price

## Import Libraries and Setup

In [3]:
import numpy as np
import pandas as pd

pd.set_option('display.max_columns',None)

In [4]:
# Expect an existing Snowpark session named `session` in Snowflake worksheets.

In [5]:
# Expect an existing Snowpark session named `session` in Snowflake worksheets.

###
### Consolidated Volume

Instruments are used to group equivalent Listings together so that the can be easily compared. Fungible listings traded in the same currency are grouped in the same Instrument. For equities, this means grouping by fungibility, based on the primary market and currency. 

Note that an Instrument is not directly queryable in reference data - querying for an object_type equals Instrument will provide all Listings that have the same Instrument Id. However, certain analytics from the BMLL Data Feed are available using the instrument.

We utilise reference data fields such this InstrumentId or the ISIN to source consolidated volume using our classified trades time_series api.

In [6]:
sql = """
SELECT * FROM "PROD_DERIVED_DATA_DB"."PUBLIC"."REFERENCE" WHERE "TICKER" = 'VOD' AND "MIC" = 'XLON'
"""
_sf_df = session.sql(sql).to_pandas()
_sf_df

,RUN_ID,LOADED_AT,REFERENCE_DATE,LISTING_ID,INSTRUMENT_ID,CURRENCY_CODE,DESCRIPTION,DISPLAY_NAME,ISIN,INSTRUMENT_TYPE,IS_ALIVE,IS_PRIMARY,MIC,MARKET_DATA_LEVEL,OPOL,TICKER,FIGI,FIGI_COMPOSITE,FIGI_SHARE_CLASS,OPERATING_MIC,REUTERS_INSTRUMENT_ID,REUTERS_ORGANIZATION_ID,REUTERS_QUOTE_ID,SEGMENT_CODE
0,reference_data_extract_scheduled__2024-12-13T1...,2024-12-14 01:16:52.049,2024-12-13,121317,121429,GBp,VODAFONE GRP.,VODAFONE GRP.,GB00BH4HKS39,Equity,True,True,XLON,L3,XLON,VOD,BBG000C6K6G9,BBG000C6K5W3,BBG001S6PJ31,None,262356.0,10749.0,1049261.0,SET1
1,reference_data_extract_scheduled__2024-12-20T1...,2024-12-21 01:26:05.391,2024-12-20,121317,121429,GBp,VODAFONE GRP.,VODAFONE GRP.,GB00BH4HKS39,Equity,True,True,XLON,L3,XLON,VOD,BBG000C6K6G9,BBG000C6K5W3,BBG001S6PJ31,None,262356.0,10749.0,1049261.0,SET1
2,reference_data_extract_scheduled__2025-01-01T1...,2025-01-02 01:15:13.422,2025-01-01,121317,121429,GBp,VODAFONE GRP.,VODAFONE GRP.,GB00BH4HKS39,Equity,True,True,XLON,L3,XLON,VOD,BBG000C6K6G9,BBG000C6K5W3,BBG001S6PJ31,None,262356.0,10749.0,1049261.0,SET1
3,reference_data_extract_scheduled__2025-01-02T1...,2025-01-03 01:16:41.215,2025-01-02,121317,121429,GBp,VODAFONE GRP.,VODAFONE GRP.,GB00BH4HKS39,Equity,True,True,XLON,L3,XLON,VOD,BBG000C6K6G9,BBG000C6K5W3,BBG001S6PJ31,None,262356.0,10749.0,1049261.0,SET1
4,reference_data_extract_scheduled__2025-01-06T1...,2025-01-07 01:16:50.001,2025-01-06,121317,121429,GBp,VODAFONE GRP.,VODAFONE GRP.,GB00BH4HKS39,Equity,True,True,XLON,L3,XLON,VOD,BBG000C6K6G9,BBG000C6K5W3,BBG001S6PJ31,None,262356.0,10749.0,1049261.0,SET1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3168,reference_data_extract_scheduled__2025-08-10T1...,2025-08-11 01:17:35.452,2025-08-10,121317,121429,GBp,VODAFONE GRP.,VODAFONE GRP.,GB00BH4HKS39,Equity,True,True,XLON,L3,XLON,VOD,BBG000C6K6G9,BBG000C6K5W3,BBG001S6PJ31,None,262356.0,10749.0,1049261.0,SET1
3169,reference_data_extract_scheduled__2025-09-15T1...,2025-09-16 01:29:52.957,2025-09-15,121317,121429,GBp,VODAFONE GRP.,VODAFONE GRP.,GB00BH4HKS39,Equity,True,True,XLON,L3,XLON,VOD,BBG000C6K6G9,BBG000C6K5W3,BBG001S6PJ31,None,262356.0,10749.0,1049261.0,SET1
3170,reference_data_extract_scheduled__2025-09-27T1...,2025-09-28 01:17:36.600,2025-09-27,121317,121429,GBp,VODAFONE GRP.,VODAFONE GRP.,GB00BH4HKS39,Equity,True,True,XLON,L3,XLON,VOD,BBG000C6K6G9,BBG000C6K5W3,BBG001S6PJ31,None,262356.0,10749.0,1049261.0,SET1
3171,reference_data_extract_scheduled__2025-11-02T1...,2025-11-03 01:17:46.476,2025-11-02,121317,121429,GBp,VODAFONE GRP.,VODAFONE GRP.,GB00BH4HKS39,Equity,True,True,XLON,L3,XLON,VOD,BBG000C6K6G9,BBG000C6K5W3,BBG001S6PJ31,None,262356.0,10749.0,1049261.0,SET1


In [7]:
ct_sql = """SELECT * FROM "PROD_DERIVED_DATA_DB"."PUBLIC"."DAILY_CLASSIFIED_TRADES" WHERE "INSTRUMENT_ID" = 121429 AND "TRADE_DATE" BETWEEN '2025-01-01' AND '2025-01-10'"""
ct = session.sql(ct_sql).to_pandas()
ct

,RUN_ID,LOADED_AT,LISTING_ID,TRADE_DATE,PUBLICATION_DATE,EXECUTION_VENUE,CLASSIFICATION_NAME,TRADE_VOLUME,TRADE_COUNT,TRADE_NOTIONAL,CURRENCY_CODE,TRADE_NOTIONAL_USD,TRADE_NOTIONAL_EUR,INSTRUMENT_ID,MIC,OPOL,TICKER,FILE_NAME,FILE_LOADED_AT
0,backfill_daily-classified-trades-CHIX_2025-01-...,2025-11-07 14:10:10.812,121649,2025-01-08,2025-01-08,CHIX,LIT_CONTINUOUS,10440834,3196,6.953489e+08,GBp,8.575661e+06,8.337217e+06,121429,CHIX,XLON,VODl,None,NaT
1,backfill_daily-classified-trades-CHIX_2025-01-...,2025-11-07 14:10:10.812,121649,2025-01-08,2025-01-08,CHIX,DARK_BELOW_LIS,2683650,290,1.785759e+08,GBp,2.202357e+06,2.141121e+06,121429,CHIX,XLON,VODl,None,NaT
2,backfill_daily-classified-trades-CHIX_2025-01-...,2025-11-07 14:10:10.812,121649,2025-01-02,2025-01-02,CHIX,DARK_BELOW_LIS,281933,42,1.939898e+07,GBp,2.408827e+05,2.333909e+05,121429,CHIX,XLON,VODl,None,NaT
3,backfill_daily-classified-trades-CHIX_2025-01-...,2025-11-07 14:10:10.812,121649,2025-01-02,2025-01-02,CHIX,LIT_CONTINUOUS,2868506,1320,1.972565e+08,GBp,2.449390e+06,2.373210e+06,121429,CHIX,XLON,VODl,None,NaT
4,backfill_daily-classified-trades-TRQX_2025-01-...,2025-11-07 23:22:35.010,121765,2025-01-06,2025-01-06,TRQX,LIT_CONTINUOUS,1181667,580,8.062475e+07,GBp,1.011569e+06,9.702370e+05,121429,TRQX,XLON,VODl,None,NaT
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
268,backfill_daily-classified-trades-BOTC_2025-01-...,2025-11-07 23:58:43.922,325319,2025-01-07,2025-01-07,SINT,SI_ADDRESSABLE_ABOVE_LIS,2390270,2,1.607068e+08,GBp,2.014383e+06,1.938211e+06,121429,BOTC,XLON,VODl,None,NaT
269,backfill_daily-classified-trades-BOTC_2025-01-...,2025-11-07 23:58:43.922,325319,2025-01-07,2025-01-07,SINT,SI_NON_ADDRESSABLE_BELOW_LIS,3258326,35,2.198452e+08,GBp,2.755655e+06,2.651453e+06,121429,BOTC,XLON,VODl,None,NaT
270,backfill_daily-classified-trades-BOTC_2025-01-...,2025-11-07 23:58:43.922,325319,2025-01-07,2025-01-07,XPAL,AUCTION_ON_DEMAND,3674,4,2.475841e+05,GBp,3.103349e+03,2.985999e+03,121429,BOTC,XLON,VODl,None,NaT
271,backfill_daily-classified-trades-BOTC_2025-01-...,2025-11-07 23:58:43.922,325319,2025-01-07,2025-01-07,LIQU,DARK_ABOVE_LIS,1000000,1,6.790000e+07,GBp,8.510941e+05,8.189109e+05,121429,BOTC,XLON,VODl,None,NaT


In [8]:
isin_sql = """SELECT DISTINCT "INSTRUMENT_ID" FROM "PROD_DERIVED_DATA_DB"."PUBLIC"."REFERENCE" WHERE "ISIN" IN ('GB00BH4HKS39')"""
isin_iids = session.sql(isin_sql).to_pandas()["INSTRUMENT_ID"].dropna().tolist()
if isin_iids:
    iid_list = ", ".join(str(i) for i in isin_iids)
    ct_sql = f"""
        SELECT *
        FROM "PROD_DERIVED_DATA_DB"."PUBLIC"."DAILY_CLASSIFIED_TRADES"
        WHERE INSTRUMENT_ID IN ({iid_list})
          AND TRADE_DATE BETWEEN '2025-01-01' AND '2025-01-10'
    """
    ct = session.sql(ct_sql).to_pandas()
else:
    ct = pd.DataFrame()
ct

,RUN_ID,LOADED_AT,LISTING_ID,TRADE_DATE,PUBLICATION_DATE,EXECUTION_VENUE,CLASSIFICATION_NAME,TRADE_VOLUME,TRADE_COUNT,TRADE_NOTIONAL,CURRENCY_CODE,TRADE_NOTIONAL_USD,TRADE_NOTIONAL_EUR,INSTRUMENT_ID,MIC,OPOL,TICKER,FILE_NAME,FILE_LOADED_AT
0,backfill_daily-classified-trades-TRQX_2025-01-...,2025-11-07 23:22:35.010,121765,2025-01-02,2025-01-02,TRQM,DARK_BELOW_LIS,184800,30,12703483.03,GBp,157742.785381,152836.726461,121429,TRQX,XLON,VODl,None,NaT
1,backfill_daily-classified-trades-TRQX_2025-01-...,2025-11-07 23:22:35.010,121765,2025-01-02,2025-01-02,TRQM,DARK_CONDITIONAL_BELOW_LIS,546838,1,37633391.16,GBp,467304.591259,452770.653288,121429,TRQX,XLON,VODl,None,NaT
2,backfill_daily-classified-trades-TRQX_2025-01-...,2025-11-07 23:22:35.010,121765,2025-01-02,2025-01-02,TRQX,LIT_CONTINUOUS,636711,399,43805722.36,GBp,543948.194708,527030.515171,121429,TRQX,XLON,VODl,None,NaT
3,backfill_daily-classified-trades-TRQX_2025-01-...,2025-11-07 23:22:35.010,121765,2025-01-02,2025-01-02,TRQA,AUCTION_ON_DEMAND,57470,14,3959083.29,GBp,49161.070570,47632.080777,121429,TRQX,XLON,VODl,None,NaT
4,backfill_daily-classified-trades-TRQX_2025-01-...,2025-11-07 23:22:35.010,121765,2025-01-06,2025-01-06,TRQA,AUCTION_ON_DEMAND,123183,22,8390657.56,GBp,105274.490025,100973.038581,121429,TRQX,XLON,VODl,None,NaT
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
306,backfill_daily-classified-trades-SGMX_2025-01-...,2025-11-07 23:32:16.837,264781778,2025-01-03,2025-01-03,SGMX,DARK_BELOW_LIS,235336,9,16336353.35,GBp,202725.655358,196840.135313,121429,SGMX,XLON,VODl,None,NaT
307,backfill_daily-classified-trades-SGMX_2025-01-...,2025-11-07 23:32:16.837,264781778,2025-01-09,2025-01-09,SGMY,AUCTION_ON_DEMAND,58994,10,3940994.06,GBp,48458.313989,47024.079563,121429,SGMX,XLON,VODl,None,NaT
308,backfill_daily-classified-trades-SGMX_2025-01-...,2025-11-07 23:32:16.837,264781778,2025-01-09,2025-01-09,SGMX,DARK_BELOW_LIS,67340,3,4497776.55,GBp,55304.490440,53667.627792,121429,SGMX,XLON,VODl,None,NaT
309,backfill_daily-classified-trades-SGMX_2025-01-...,2025-11-07 23:32:16.837,264781778,2025-01-07,2025-01-07,SGMY,AUCTION_ON_DEMAND,31870,15,2151208.07,GBp,26964.367692,25944.739432,121429,SGMX,XLON,VODl,None,NaT


###
### Consolidated BBO: Precompted
We provide access to a precomputed 10 level consolidated BBO (either NBBO in the US or EBBO in Europe), with price levels, and aggregated sizes for a instrument across all listing venues.

In [9]:
sql = """
SELECT * FROM "PROD_L2_CBBO_DB"."PUBLIC"."VW_CBBO_EQUITY" WHERE "MIC" = 'XLON' AND "TICKER" = 'VOD' AND "TRADE_DATE" = '2024-11-01' ORDER BY "EVENT_TIMESTAMP" LIMIT 5
"""
_sf_df = session.sql(sql).to_pandas()
_sf_df

,TICKER,ISO_EXCHANGE_CODE,TRADE_DATE,EVENT_TIMESTAMP,LOCAL_TIMESTAMP,TIMESTAMP_NANOSECONDS,TZ_OFFSET,BID_PRICE1,BID_QUANTITY1,BID_NUM_ORDERS1,ASK_PRICE1,ASK_QUANTITY1,ASK_NUM_ORDERS1,BID_PRICE2,BID_QUANTITY2,BID_NUM_ORDERS2,ASK_PRICE2,ASK_QUANTITY2,ASK_NUM_ORDERS2,BID_PRICE3,BID_QUANTITY3,BID_NUM_ORDERS3,ASK_PRICE3,ASK_QUANTITY3,ASK_NUM_ORDERS3,BID_PRICE4,BID_QUANTITY4,BID_NUM_ORDERS4,ASK_PRICE4,ASK_QUANTITY4,ASK_NUM_ORDERS4,BID_PRICE5,BID_QUANTITY5,BID_NUM_ORDERS5,ASK_PRICE5,ASK_QUANTITY5,ASK_NUM_ORDERS5,BID_PRICE6,BID_QUANTITY6,BID_NUM_ORDERS6,ASK_PRICE6,ASK_QUANTITY6,ASK_NUM_ORDERS6,BID_PRICE7,BID_QUANTITY7,BID_NUM_ORDERS7,ASK_PRICE7,ASK_QUANTITY7,ASK_NUM_ORDERS7,BID_PRICE8,BID_QUANTITY8,BID_NUM_ORDERS8,ASK_PRICE8,ASK_QUANTITY8,ASK_NUM_ORDERS8,BID_PRICE9,BID_QUANTITY9,BID_NUM_ORDERS9,ASK_PRICE9,ASK_QUANTITY9,ASK_NUM_ORDERS9,BID_PRICE10,BID_QUANTITY10,BID_NUM_ORDERS10,ASK_PRICE10,ASK_QUANTITY10,ASK_NUM_ORDERS10,BID_LEVEL_COUNT,ASK_LEVEL_COUNT,L2_EVENT_NO,MIC,EXCHANGE_TICKER,BMLL_OBJECT_ID,CURRENCY_CODE,LOADED_AT
0,VOD,LON,2024-11-01,2024-11-01 08:00:00.148,2024-11-01 08:00:00.148,1730448000148000000,0,68.9,700,1,78.88,4000,1,68.86,1266,1,78.92,1043,1,NaN,0,0,NaN,0,0,NaN,0,0,NaN,0,0,NaN,0,0,NaN,0,0,NaN,0,0,NaN,0,0,NaN,0,0,NaN,0,0,NaN,0,0,NaN,0,0,NaN,0,0,NaN,0,0,NaN,0,0,NaN,0,0,2,2,1,XLON,VOD,121317,GBp,2026-07-22 16:07:38.102
1,VOD,LON,2024-11-01,2024-11-01 08:00:00.159,2024-11-01 08:00:00.159,1730448000159000000,0,68.9,700,1,78.88,4000,1,68.86,2312,2,78.92,1043,1,NaN,0,0,NaN,0,0,NaN,0,0,NaN,0,0,NaN,0,0,NaN,0,0,NaN,0,0,NaN,0,0,NaN,0,0,NaN,0,0,NaN,0,0,NaN,0,0,NaN,0,0,NaN,0,0,NaN,0,0,NaN,0,0,2,2,2,XLON,VOD,121317,GBp,2026-07-22 16:07:38.102
2,VOD,LON,2024-11-01,2024-11-01 08:00:00.171,2024-11-01 08:00:00.171,1730448000171000000,0,68.9,700,1,78.88,4000,1,68.86,2312,2,78.90,1254,1,NaN,0,0,78.92,2242,2,NaN,0,0,NaN,0,0,NaN,0,0,NaN,0,0,NaN,0,0,NaN,0,0,NaN,0,0,NaN,0,0,NaN,0,0,NaN,0,0,NaN,0,0,NaN,0,0,NaN,0,0,NaN,0,0,2,3,3,XLON,VOD,121317,GBp,2026-07-22 16:07:38.102
3,VOD,LON,2024-11-01,2024-11-01 08:00:00.220,2024-11-01 08:00:00.220,1730448000220000000,0,68.9,700,1,78.88,4000,1,68.86,1266,1,78.90,1254,1,NaN,0,0,78.92,2242,2,NaN,0,0,NaN,0,0,NaN,0,0,NaN,0,0,NaN,0,0,NaN,0,0,NaN,0,0,NaN,0,0,NaN,0,0,NaN,0,0,NaN,0,0,NaN,0,0,NaN,0,0,NaN,0,0,2,3,4,XLON,VOD,121317,GBp,2026-07-22 16:07:38.102
4,VOD,LON,2024-11-01,2024-11-01 08:00:00.291,2024-11-01 08:00:00.291,1730448000291000000,0,68.9,700,1,78.88,4000,1,68.86,1266,1,78.92,2242,2,NaN,0,0,NaN,0,0,NaN,0,0,NaN,0,0,NaN,0,0,NaN,0,0,NaN,0,0,NaN,0,0,NaN,0,0,NaN,0,0,NaN,0,0,NaN,0,0,NaN,0,0,NaN,0,0,NaN,0,0,NaN,0,0,2,2,5,XLON,VOD,121317,GBp,2026-07-22 16:07:38.102


###
### Consolidated BBO: On the fly
It is also possible by utilising our granular data to create your own consolidated book, with price levels, and aggregated sizes for a instrument across all listing venues.

## Building CBBO On-the-Fly

While BMLL provides precomputed 10-level consolidated books, we can also build custom consolidations
from granular per-venue BBO data. This gives flexibility to handle:
- Custom security universes
- Non-standard consolidation rules
- Historical periods before precomputed CBBO was available
- Specific use cases requiring different consolidation logic

### Helper Functions for CBBO Construction

## Main CBBO Consolidation Algorithm

### Algorithm Overview

The consolidation process:
1. **Fetch per-venue BBO**: Get best bid/ask from each venue separately
2. **Timestamp alignment**: Ensure all quotes are aligned to the same timeline
3. **Forward-fill gaps**: BBO doesn't update if there's no change; fill down last-seen price
4. **Compute consolidated prices**:
   - Best bid = Max(all bids) - highest price buyers willing to pay
   - Best ask = Min(all asks) - lowest price sellers willing to accept
5. **Aggregate sizes**: Sum sizes from all venues at the consolidated best prices

### Key Challenges Handled

- **Index alignment**: Venues have different quote timing; use forward-fill to align
- **Duplicate timestamps**: Some timestamps may repeat; use last-observation-carry-forward
- **Stale data**: When no new quotes, price remains valid until next update
- **Multiple venues at best**: Sum sizes when multiple venues share the best bid/ask

In [10]:
def merge_duplicate_mics(bbos, mics):
    if len(mics) == len(set(mics)):
        return bbos, mics
    new_bbos = []
    seen = set()
    for mic in set(mics):
        indices = [i for i, m in enumerate(mics) if m == mic]
        merged = pd.concat([bbos[i] for i in indices])
        new_bbos.append(merged)
        seen.add(mic)
    return new_bbos, list(seen)

In [11]:
def get_cbbo(iid, date, return_all=True):
    ref_sql = f"""
        SELECT LISTING_ID, MIC
        FROM "PROD_DERIVED_DATA_DB"."PUBLIC"."REFERENCE"
        WHERE INSTRUMENT_ID = {iid}
          AND REFERENCE_DATE = '{date}'
    """
    ref_df = session.sql(ref_sql).to_pandas()
    if ref_df.empty:
        return pd.DataFrame()

    listing_ids = ref_df["LISTING_ID"].tolist()
    lid_to_mic = dict(zip(ref_df["LISTING_ID"], ref_df["MIC"]))
    lid_list = ", ".join(str(l) for l in listing_ids)

    l2_sql = f"""
        SELECT
            BMLL_OBJECT_ID AS LISTING_ID,
            EVENT_TIMESTAMP,
            BID_PRICE1,
            BID_QUANTITY1,
            ASK_PRICE1,
            ASK_QUANTITY1
        FROM "PROD_L2_CBBO_DB"."PUBLIC"."VW_L2_DATA_EQUITY"
        WHERE BMLL_OBJECT_ID IN ({lid_list})
          AND TRADE_DATE = '{date}'
        ORDER BY EVENT_TIMESTAMP
    """
    l2 = session.sql(l2_sql).to_pandas()
    if l2.empty:
        return pd.DataFrame()

    bbos = []
    mics = []
    for lid, group in l2.groupby("LISTING_ID"):
        mic = lid_to_mic.get(lid, f"LID_{lid}")
        df = group.rename(columns={
            "EVENT_TIMESTAMP": "event_timestamp",
            "BID_PRICE1": "best_bid_price",
            "BID_QUANTITY1": "best_bid_size",
            "ASK_PRICE1": "best_ask_price",
            "ASK_QUANTITY1": "best_ask_size",
        })[["event_timestamp", "best_bid_price", "best_ask_price", "best_bid_size", "best_ask_size"]]
        df = df.set_index("event_timestamp").rename(columns={
            "best_bid_price": f"best_bid_price_{mic}",
            "best_bid_size": f"best_bid_size_{mic}",
            "best_ask_price": f"best_ask_price_{mic}",
            "best_ask_size": f"best_ask_size_{mic}",
        })
        bbos.append(df.replace(np.nan, 0))
        mics.append(mic)

    bbos, mics = merge_duplicate_mics(bbos, mics)
    try:
        bbo_df = pd.concat(bbos, axis=1, sort=False)
    except Exception:
        bbos = [df.loc[~df.index.duplicated(keep="last")] for df in bbos]
        bbo_df = pd.concat(bbos, axis=1, sort=False)

    best_cols = [col for col in bbo_df.columns if "best_" in col]
    if not bbo_df.empty:
        bbo_df[best_cols] = bbo_df[best_cols].ffill()
    bbo_df = bbo_df.replace(0, np.nan)

    for mic in mics:
        bbo_df[f"best_bid_size_{mic}"] = bbo_df[f"best_bid_size_{mic}"].fillna(0)
        bbo_df[f"best_ask_size_{mic}"] = bbo_df[f"best_ask_size_{mic}"].fillna(0)

    bbo_df["cbb_price"] = bbo_df.filter(like="best_bid_price").max(axis=1)
    bbo_df["cbo_price"] = bbo_df.filter(like="best_ask_price").min(axis=1)
    bbo_df["cbo_size"] = 0
    bbo_df["cbb_size"] = 0

    for mic in mics:
        bbo_df["cbo_size"] += (bbo_df[f"best_ask_price_{mic}"] == bbo_df["cbo_price"]) * bbo_df[f"best_ask_size_{mic}"]
        bbo_df["cbb_size"] += (bbo_df[f"best_bid_price_{mic}"] == bbo_df["cbb_price"]) * bbo_df[f"best_bid_size_{mic}"]

    bbo_df = bbo_df.reset_index().drop_duplicates(subset=bbo_df.columns.difference(['event_timestamp']), keep='first')
    return bbo_df if return_all else bbo_df[['event_timestamp', "cbb_size", "cbb_price", "cbo_price", "cbo_size"]]

In [12]:
iid = 121429
ref = session.sql(f"SELECT * FROM REFERENCE WHERE INSTRUMENT_ID = {iid} AND REFERENCE_DATE = '2024-07-02'").to_pandas()

BAH = get_cbbo(iid, '2024-07-02', return_all=False)
BAH

,event_timestamp,cbb_size,cbb_price,cbo_price,cbo_size
0,2024-07-02 03:20:17.002480,450.0,70.00,NaN,0.0
1,2024-07-02 04:00:03.610308,450.0,70.00,NaN,0.0
15,2024-07-02 04:00:03.610426,450.0,70.00,70.50,22000.0
35,2024-07-02 05:30:08.193980,3000.0,72.00,70.50,22000.0
36,2024-07-02 06:30:09.955497,3000.0,72.00,70.50,22000.0
...,...,...,...,...,...
296996,2024-07-02 16:30:00.069985,5037.0,68.98,69.02,14874.0
296997,2024-07-02 16:30:00.069987,5037.0,68.98,69.02,14874.0
296998,2024-07-02 16:30:00.069988,5037.0,68.98,69.02,14874.0
296999,2024-07-02 16:30:00.069990,5037.0,68.98,69.02,14874.0


In [13]:
get_cbbo(iid, '2024-07-02', return_all=True).head()

,event_timestamp,best_bid_price_LID_121317,best_ask_price_LID_121317,best_bid_size_LID_121317,best_ask_size_LID_121317,best_bid_price_LID_121648,best_ask_price_LID_121648,best_bid_size_LID_121648,best_ask_size_LID_121648,best_bid_price_LID_121649,best_ask_price_LID_121649,best_bid_size_LID_121649,best_ask_size_LID_121649,best_bid_price_LID_121765,best_ask_price_LID_121765,best_bid_size_LID_121765,best_ask_size_LID_121765,best_bid_price_LID_292893575,best_ask_price_LID_292893575,best_bid_size_LID_292893575,best_ask_size_LID_292893575,best_bid_price_LID_321638,best_ask_price_LID_321638,best_bid_size_LID_321638,best_ask_size_LID_321638,cbb_price,cbo_price,cbo_size,cbb_size
0,2024-07-02 03:20:17.002480,NaN,NaN,0.0,0.0,NaN,NaN,0.0,0.0,NaN,NaN,0.0,0.0,NaN,NaN,0.0,0.0,70.0,NaN,450.0,0.0,NaN,NaN,0.0,0.0,70.0,NaN,0.0,450.0
1,2024-07-02 04:00:03.610308,69.24,NaN,35000.0,0.0,NaN,NaN,0.0,0.0,NaN,NaN,0.0,0.0,NaN,NaN,0.0,0.0,70.0,NaN,450.0,0.0,NaN,NaN,0.0,0.0,70.0,NaN,0.0,450.0
15,2024-07-02 04:00:03.610426,69.24,70.5,35000.0,22000.0,NaN,NaN,0.0,0.0,NaN,NaN,0.0,0.0,NaN,NaN,0.0,0.0,70.0,NaN,450.0,0.0,NaN,NaN,0.0,0.0,70.0,70.5,22000.0,450.0
35,2024-07-02 05:30:08.193980,69.24,70.5,35000.0,22000.0,NaN,NaN,0.0,0.0,NaN,NaN,0.0,0.0,NaN,NaN,0.0,0.0,72.0,NaN,3000.0,0.0,NaN,NaN,0.0,0.0,72.0,70.5,22000.0,3000.0
36,2024-07-02 06:30:09.955497,69.24,70.5,35000.0,22000.0,NaN,NaN,0.0,0.0,NaN,NaN,0.0,0.0,NaN,NaN,0.0,0.0,72.0,72.92,3000.0,5000.0,NaN,NaN,0.0,0.0,72.0,70.5,22000.0,3000.0
